## Datos ordenador

- Cores: 16
- Hilos: 24
- Infraestructura: Docker

In [2]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "BotNet3")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/18 19:10:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/18 19:10:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
import numpy as np
import random
from tqdm.notebook import trange, tqdm
from copy import deepcopy

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def readFile (filename): 
    """
    Arguments: 
        filename – name of the spam dataset file 
        12 columns: 11 features/dimensions (X) + 1 column with labels (Y) 
        Y -- Train labels (0 if normal traffic, 1 if botnet)  
        m rows: number of examples (m) 
    Returns: 
        An RDD containing the data of filename. Each example (row) of the file 
        corresponds to one RDD record. Each record of the RDD is a tuple (X,y).  
        “X” is an array containing the 11 features (float number) of an example  
        “y” is the 12th column of an example (integer 0/1)  
    """
    return sc.textFile(filename).map(lambda line: np.array(line.split(","), dtype=float)).map(lambda x: (x[:-1], x[-1]))



def normalize (RDD_Xy): 
    """
    Arguments: 
        RDD_Xy is an RDD containing data examples. Each record of the RDD is a tuple 
        (X,y). 
        “X” is an array containing the 11 features (float number) of an example 
        “y” is the label of the example (integer 0/1)  
    Returns: 
        An RDD rescaled to N(0,1) in each column (mean=0, standard deviation=1) 
    """
    mean_rdd = RDD_Xy.map(lambda x: ("key", (x[0], 1, x[0] ** 2))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1], a[2] + b[2])).mapValues(lambda x: (x[0] / x[1], x[2] / x[1]))
    stats = mean_rdd.first()[1]
    mean = stats[0]
    var = stats[1] - (mean ** 2)
    std = np.sqrt(var)
    rdd = RDD_Xy.map(lambda xy: ((xy[0] - mean)/std, xy[1]))
    return rdd


def train (RDD_Xy, iterations, learning_rate, lambda_reg=1): 
    """
    Arguments: 
        RDD_Xy --- RDD containing data examples. Each record of the RDD is a tuple 
        (X,y). 
        “X” is an array containing the 11 features (float number) of an example 
        “y” is the label of the example (integer 0/1)  
        iterations -- number of iterations of the optimization loop 
        learning_rate -- learning rate of the gradient descent 
        lambda_reg –- regularization rate 
    Returns: 
        A list or array containing the weights “w” and bias “b”	at the end of the 
        training process	
    """
    n_rows = RDD_Xy.count()
    n_cols = len(RDD_Xy.first()[0])

    ws = np.random.random(n_cols)
    b = np.random.random()

    for _ in tqdm(range(iterations)):
        # X, y, pred
        rdd_preds = RDD_Xy.map(lambda xy: (xy[0], xy[1], predict(ws, b, xy[0])))
        #rdd_preds.cache()

        # Tamaño del paso
        dws = rdd_preds.map(lambda xyp: (xyp[2] - xyp[1]) * xyp[0]).reduce(lambda a, b: a + b)
        dws = dws/n_rows + lambda_reg/n_cols * ws

        db = rdd_preds.map(lambda xyp: xyp[2] - xyp[1]).reduce(lambda a, b: a + b)
        db = db/n_rows

        # Actualizar los pesos
        ws = ws - learning_rate* dws
        b = b - learning_rate * db
    return ws, b


def predict (w, b, X): 
    """
    Arguments: 
        w -- weights 
        b -- bias 
        X –- Example to be predicted  
    Returns: 
        Y_pred –- a value (0/1) corresponding to the prediction of X  
    """
    prob = sigmoid(np.dot(w, X) + b)

    return 1 if prob >= 0.5 else 0


def accuracy (w, b, RDD_Xy): 
    """
    Arguments: 
        w -- weights 
        b -- bias 
        RDD_Xy –- RDD containing examples to be predicted  
    Returns: 
        accuracy -- the number of predictions that are correct divided by the number         
        of records (examples) in RDD_xy.  
        Predict function can be used for predicting a single example
    """
    preds = RDD_Xy.map(lambda xy: ("key",(xy[1] == predict(w, b, xy[0]), 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).mapValues(lambda x: x[0] / x[1]).first()[1]

    return preds * 100

def transform(rdd_data, num_blocks_cv):
    return rdd_data.map(lambda xy: (np.random.randint(num_blocks_cv), xy)).cache()
    

def get_block_data(rdd_data, i):
    train_data = rdd_data.flatMap(lambda x: [x[1]] if x[0] != i else [])
    test_data = rdd_data.flatMap(lambda x: [x[1]] if x[0] == i else [])
    return train_data, test_data

In [ ]:
import time

# read data
path = "/opt/local/practica_spark/datos/botnet_tot_syn_l.csv"
data = readFile(path)
#data.cache()
# standardize
data = normalize(data)
num_blocks_cv = 10
nIter = 3
learningRate = 1e-4
# Shuffle rows and transform data, specifying the number of blocks
data_cv = transform(data, num_blocks_cv)
accs = []


print(data_cv.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a+b).collect())
for i in range(num_blocks_cv):
    tr_data, test_data = get_block_data(data_cv, i)
    #print(tr_data.count(), test_data.count())
    #print(tr_data.take(1), test_data.take(1))
    start = time.time()
    ws, b = train(tr_data, nIter, learningRate)
    
    
    end = time.time()
    print(f'Total time : {end - start:.4f}s')
    acc = accuracy(ws, b, test_data)
    accs.append(acc)
    print("acc:", acc)
    
print("average acc:", np.mean(accs))

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 9) / 9]
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

Exception in thread "serve RDD 6 with partitions 0" java.net.SocketTimeoutException: Accept timed out
	at java.base/java.net.PlainSocketImpl.socketAccept(Native Method)
	at java.base/java.net.AbstractPlainSocketImpl.accept(AbstractPlainSocketImpl.java:474)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:565)
	at java.base/java.net.ServerSocket.accept(ServerSocket.java:533)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)


In [7]:
import matplotlib.pyplot as plt 
def plot(x, y):
    plt.plot(x, y)
    plt.show()

## Experimento 1

In [6]:
learningRates = [5, 3, 2, 1, 1e-1, 1e-2, 1e-3]
regularizaciones = [3, 2, 1, 0.5, 0.1, 0.001, 0]